## Gold Layer — Dimensional Model (Star Schema)

We build three dimension tables (Airline, Airport, Date) and one fact table 
(FactFlight), reading from the Silver layer. Each dimension uses a 
surrogate key that increments across loads and never resets — this is what 
lets the fact table safely reference dimension rows without those 
references breaking on future loads.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df_silver_flight = spark.table("aviation_ws.silver.flight_data")
df_silver_flight.limit(10).display()

YEAR,MONTH,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,CANCELLATION_CODE,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,IngestionID,SourceFileName,SourceFileHash,LoadedAt,CANCELLED,DIVERTED,FlightKey,FlightKeyHash,RowChanged,RowChangedHash
2026,1,2026-01-02,DL,412,12451,JAX,"Jacksonville, FL",Florida,12953,LGA,"New York, NY",New York,1205,1234,29.00,29.00,1418,1439,21.00,21.00,null,133.00,125.00,108.00,834.00,0.00,0.00,0.00,0.00,21.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z,false,false,2026-01-02||DL||412||JAX||LGA||1205,00b54af791e849e9590d8a06355b81df0269c4845bee120577179913473910ed,1234||29.00||1439||21.00||false||false||0.00||0.00||0.00||0.00||21.00,86006f721f0b6f2f89cd4552323e31dd722a0fd5ab09309bf1c5dc85fb81412d
2026,1,2026-01-02,DL,413,12892,LAX,"Los Angeles, CA",California,13487,MSP,"Minneapolis, MN",Minnesota,1155,1146,-9.00,0.00,1735,1723,-12.00,0.00,null,220.00,217.00,190.00,1535.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z,false,false,2026-01-02||DL||413||LAX||MSP||1155,42a19c0d1195b2a4b316b0fe3f30105ef232039f66ff97ac5a6379fda203b839,1146||-9.00||1723||-12.00||false||false||NULL||NULL||NULL||NULL||NULL,7833ce3bf6e83392c28f7589a25c4ff84e93822265d41befef45cddb2f53a3dc
2026,1,2026-01-02,DL,414,10397,ATL,"Atlanta, GA",Georgia,11298,DFW,"Dallas/Fort Worth, TX",Texas,1100,1141,41.00,41.00,1231,1301,30.00,30.00,null,151.00,140.00,112.00,731.00,30.00,0.00,0.00,0.00,0.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z,false,false,2026-01-02||DL||414||ATL||DFW||1100,1502d6359fb809206080d303f10121dbca5610ac9bb4786fb48427d4d4c40ab5,1141||41.00||1301||30.00||false||false||30.00||0.00||0.00||0.00||0.00,0c018f94b5fdd5e9dd1f2ddd65ad25b7a683863198303daad1e998cdd4861a4a
2026,1,2026-01-02,DL,415,14747,SEA,"Seattle, WA",Washington,12478,JFK,"New York, NY",New York,1136,1218,42.00,42.00,2007,2039,32.00,32.00,null,331.00,321.00,276.00,2422.00,1.00,0.00,0.00,0.00,31.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z,false,false,2026-01-02||DL||415||SEA||JFK||1136,9ad727283558ad9f5cf04d2528f8b1c53fdfaa6f0fbc91c7176c35abd85aaac0,1218||42.00||2039||32.00||false||false||1.00||0.00||0.00||0.00||31.00,d5f13bf6fa0db6328710bbdefe1b05f009cb7be363fcb7f9e0e43bf9a7247e6b
2026,1,2026-01-02,DL,416,10299,ANC,"Anchorage, AK",Alaska,10397,ATL,"Atlanta, GA",Georgia,2015,2052,37.00,37.00,718,741,23.00,23.00,null,423.00,409.00,365.00,3417.00,23.00,0.00,0.00,0.00,0.00,1,2026_01.csv,null,2026-08-10T14:30:22.577Z,false,false,2026-01-02||DL||416||ANC||ATL||2015,43d6919f5fd79e2199167fb1dc026b46f5ae331b09e6ee7ceecf35fc05527e28,2052||37.00||741||23.00||false||false||23.00||0.00||0.00||0.00||0.00,fc482f122683060804b6ffad775302178267ea8a71f5af993070612d7acefb3a
2026,1,2026-01-02,DL,416,10397,ATL,"Atlanta, GA",Georgia,10299,ANC,"Anchorage, AK",Alaska,1515,1513,-2.00,0.00,1857,1853,-4.00,0.00,null,462.00,460.00,436.00,3417.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z,false,false,2026-01-02||DL||416||ATL||ANC||1515,e203eadcf91b4d6aa76d1f26639c28c5e5406f5772917de33b70b7feecb75027,1513||-2.00||1853||-4.00||false||false||NULL||NULL||NULL||NULL||NULL,b52819ec1a8daf5edbf4fc34c93afb63c73f81f997e82b13966989a15579a3c2
2026,1,2026-01-02,DL,417,12889,LAS,"Las Vegas, NV",Nevada,10397,ATL,"Atlanta, GA",Georgia,2204,2203,-1.00,0.00,500,502,2.00,2.00,null,236.00,239.00,191.00,1747.00,null,null,null,null,null,1,2026_01.csv,null,2026-08-10T14:30:22.577Z,false,false,2026-01-02||DL||417||LAS||ATL||2204,e1fd6b53e8549a01d4cb0731977263fde1caccb07a86fbe92894a273ca28b7be,2203||-1.00||502||2.00||false||false||NULL||NULL||NULL||NULL||NULL,ab959352b7c6884e88e12517791d3fd3f25323de597aaf2f3ab07b2925be1cdf
2026,1,2026-01-02,DL,418,10849,BZN,"Bozeman, MT",Montana,10721,BOS,"

In [0]:
distinct_carrier = df_silver_flight.select(col("OP_UNIQUE_CARRIER").alias("Carrier_Code")).distinct()
distinct_carrier.display()

Carrier_Code
DL
F9
G4
MQ
NK
OH
OO
UA
WN
AA


### Step 2 — Check existing DimAirline and find the starting point for new keys

On the very first run, DimAirline doesn't exist yet, so there's nothing to 
compare against and no existing surrogate keys. On every run after that, 
we need to know two things: which carriers are already there (so we don't 
re-add them), and what the highest surrogate key currently is (so new 
carriers get keys that continue from there, never restarting at 1).

In [0]:
table_exists = spark.catalog.tableExists("aviation_ws.gold.dim_airline")
if table_exists:
    dim_airline_exist = spark.table("aviation_ws.gold.dim_airline")
    max_key_row = dim_airline_exist.agg({"AirlineKey":"max"}).collect()[0]
    max_key = max_key_row[0] if max_key_row[0] is not None else 0
else:
    dim_airline_exist = None
    max_key=0
print(f"Table exists:{table_exists}")
print(f"Current Max AirlineKey:{max_key}")

Table exists:True
Current Max AirlineKey:13


In [0]:
if dim_airline_exist is not None:
    new_carrier = distinct_carrier.join(dim_airline_exist,distinct_carrier["Carrier_Code"]==dim_airline_exist["Carrier_Code"],"left_anti")
else:
    new_carrier = distinct_carrier
window_partition = Window.orderBy("Carrier_Code")
new_carrier_with_key = new_carrier.withColumn("AirlineKey",row_number().over(window_partition)+max_key)
new_carrier_with_key.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Carrier_Code,AirlineKey


In [0]:
if not table_exists:
    new_carrier_with_key.write.format("delta").saveAsTable("aviation_ws.gold.dim_airline")
    print("DimAirline created.")
else:
    new_carrier_with_key.write.format("delta").mode("append").saveAsTable("aviation_ws.gold.dim_airline")
    print("New carriers appended to DimAirline.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


New carriers appended to DimAirline.


In [0]:
spark.table("aviation_ws.gold.dim_airline").display()

Carrier_Code,AirlineKey
AA,1
AS,2
B6,3
DL,4
F9,5
G4,6
MQ,7
NK,8
OH,9
OO,10


### DimAirport

Airports appear in two places in the flight data — as an origin and as a 
destination. We need one row per unique airport, regardless of which role 
it played in a given flight, so we first combine ORIGIN and DEST into a 
single list before finding distinct values.

In [0]:
origin_airport = df_silver_flight.select(col("ORIGIN_AIRPORT_ID").alias("AirportID"),
                                         col("ORIGIN").alias("AirportCode"),
                                         col("ORIGIN_CITY_NAME").alias("CityName"),
                                         col("ORIGIN_STATE_NM").alias("StateName"))
# origin_airport.display()

In [0]:
dest_airport = df_silver_flight.select(col("DEST_AIRPORT_ID").alias("AirportID"),
                                       col("DEST").alias("AirportCode"),
                                       col("DEST_CITY_NAME").alias("CityName"),
                                       col("DEST_STATE_NM").alias("StateName"))
dest_airport.display()

AirportID,AirportCode,CityName,StateName
12953,LGA,"New York, NY",New York
13487,MSP,"Minneapolis, MN",Minnesota
11298,DFW,"Dallas/Fort Worth, TX",Texas
12478,JFK,"New York, NY",New York
10397,ATL,"Atlanta, GA",Georgia
10299,ANC,"Anchorage, AK",Alaska
10397,ATL,"Atlanta, GA",Georgia
10721,BOS,"Boston, MA",Massachusetts
12173,HNL,"Honolulu, HI",Hawaii
11298,DFW,"Dallas/Fort Worth, TX",Texas


In [0]:
all_airport = origin_airport.union(dest_airport).distinct()
all_airport.display()

AirportID,AirportCode,CityName,StateName
12451,JAX,"Jacksonville, FL",Florida
12892,LAX,"Los Angeles, CA",California
10397,ATL,"Atlanta, GA",Georgia
14747,SEA,"Seattle, WA",Washington
10299,ANC,"Anchorage, AK",Alaska
12889,LAS,"Las Vegas, NV",Nevada
10849,BZN,"Bozeman, MT",Montana
11433,DTW,"Detroit, MI",Michigan
12173,HNL,"Honolulu, HI",Hawaii
12478,JFK,"New York, NY",New York


### Check existing DimAirport and find the starting surrogate key

Same pattern as DimAirline: check if the table already exists, and if so, 
find the current highest AirportKey so new airports continue counting up 
from there instead of restarting.

In [0]:
airport_table_exists = spark.catalog.tableExists("aviation_ws.gold.dim_airport")

if airport_table_exists:
    dim_airport_existing = spark.table("aviation_ws.gold.dim_airport")
    max_airport_key_row = dim_airport_existing.agg({"AirportKey":"max"}).collect()[0]
    max_airport_key = max_airport_key_row[0] if max_airport_key_row[0] is not None else 0
else:
    dim_airport_existing = None
    max_airport_key = 0

print(f"Table exists: {airport_table_exists}")
print(f"Current max AirportKey: {max_airport_key}")


Table exists: True
Current max AirportKey: 342


### Find genuinely new airports and assign surrogate keys

Same anti-join pattern used for DimAirline: keep only airports not already 
in DimAirport, then assign sequential keys starting from max_airport_key + 1.

In [0]:
if dim_airport_existing is not None:
    new_airport = all_airport.join(dim_airport_existing,all_airport["AirportCode"]==dim_airport_existing["AirportCode"],"left_anti")
else:
    new_airport = all_airport
airport_window_partition = Window.orderBy("AirportCode")
new_airport_with_keys = new_airport.withColumn("AirportKey",row_number().over(airport_window_partition)+max_airport_key)
new_airport_with_keys.display()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


AirportID,AirportCode,CityName,StateName,AirportKey


### Save new airports into DimAirport

Create the table on first run, append new rows on every run after — 
existing airports and their keys are never touched.

In [0]:
if not airport_table_exists:
    new_airport_with_keys.write.format("delta").saveAsTable("aviation_ws.gold.dim_airport")
    print("DimAirport created.")
else:
    new_airport_with_keys.write.format("delta").mode("append").saveAsTable("aviation_ws.gold.dim_airport")
    print("New airports appended to DimAirport.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


New airports appended to DimAirport.


In [0]:
spark.table("aviation_ws.gold.dim_airport").display()

AirportID,AirportCode,CityName,StateName,AirportKey
10135,ABE,"Allentown/Bethlehem/Easton, PA",Pennsylvania,1
10136,ABI,"Abilene, TX",Texas,2
10140,ABQ,"Albuquerque, NM",New Mexico,3
10141,ABR,"Aberdeen, SD",South Dakota,4
10155,ACT,"Waco, TX",Texas,5
10157,ACV,"Arcata/Eureka, CA",California,6
10158,ACY,"Atlantic City, NJ",New Jersey,7
10165,ADK,"Adak Island, AK",Alaska,8
10170,ADQ,"Kodiak, AK",Alaska,9
10185,AEX,"Alexandria, LA",Louisiana,10


### DimDate

Unlike Airline and Airport, a date dimension is usually built by generating 
a full calendar range up front — not by discovering dates as they appear in 
the data. This means every date in the range exists in the dimension even 
before any flight uses it, which is standard practice so date-based 
filtering/joins in Power BI never hit a missing date.

In [0]:
date_range = df_silver_flight.agg(min("FL_DATE").alias("MinDate"),max("FL_Date").alias("MaxDate")).collect()[0]
min_date = date_range["MinDate"]
max_date = date_range["MaxDate"]
print(f'The Min Date is :{min_date}')
print(f'The Max Date is : {max_date}')

The Min Date is :2026-01-01
The Max Date is : 2026-01-31


In [0]:
date_range_df = spark.range(1).select(explode(sequence(lit(min_date),lit(max_date),expr("interval 1 day"))).alias("FullDate"))
date_range_df.display()

FullDate
2026-01-01
2026-01-02
2026-01-03
2026-01-04
2026-01-05
2026-01-06
2026-01-07
2026-01-08
2026-01-09
2026-01-10


 #### .select() vs .withColumn() is just a stylistic choice for how many columns you're adding at once: .select() lets you build several new columns in one call by listing them all; .withColumn() adds one column at a time to an existing DataFrame.


In [0]:
dim_date_final = date_range_df.select(
                                col("FullDate"),
                                year("FullDate").alias("Year"),
                                month("FullDate").alias("Month"),
                                date_format("FullDate", "MMMM").alias("MonthName"),
                                dayofmonth("FullDate").alias("Day"),
                                date_format("FullDate", "EEEE").alias("DayName"),
                                quarter("FullDate").alias("Quarter"))

dim_date_final.limit(10).display()

FullDate,Year,Month,MonthName,Day,DayName,Quarter
2026-01-01,2026,1,January,1,Thursday,1
2026-01-02,2026,1,January,2,Friday,1
2026-01-03,2026,1,January,3,Saturday,1
2026-01-04,2026,1,January,4,Sunday,1
2026-01-05,2026,1,January,5,Monday,1
2026-01-06,2026,1,January,6,Tuesday,1
2026-01-07,2026,1,January,7,Wednesday,1
2026-01-08,2026,1,January,8,Thursday,1
2026-01-09,2026,1,January,9,Friday,1
2026-01-10,2026,1,January,10,Saturday,1


In [0]:
dim_date_final = dim_date_final.withColumn(
    "DateKey",
    date_format(col("FullDate"), "yyyyMMdd").cast("int")
)

dim_date_final.limit(5).display()

FullDate,Year,Month,MonthName,Day,DayName,Quarter,DateKey
2026-01-01,2026,1,January,1,Thursday,1,20260101
2026-01-02,2026,1,January,2,Friday,1,20260102
2026-01-03,2026,1,January,3,Saturday,1,20260103
2026-01-04,2026,1,January,4,Sunday,1,20260104
2026-01-05,2026,1,January,5,Monday,1,20260105


In [0]:
dim_date_final.write.format("delta").mode("overwrite").saveAsTable("aviation_ws.gold.dim_date")

print("DimDate saved.")

DimDate saved.


In [0]:
spark.table("aviation_ws.gold.dim_date").display()

FullDate,Year,Month,MonthName,Day,DayName,Quarter,DateKey
2026-01-01,2026,1,January,1,Thursday,1,20260101
2026-01-02,2026,1,January,2,Friday,1,20260102
2026-01-03,2026,1,January,3,Saturday,1,20260103
2026-01-04,2026,1,January,4,Sunday,1,20260104
2026-01-05,2026,1,January,5,Monday,1,20260105
2026-01-06,2026,1,January,6,Tuesday,1,20260106
2026-01-07,2026,1,January,7,Wednesday,1,20260107
2026-01-08,2026,1,January,8,Thursday,1,20260108
2026-01-09,2026,1,January,9,Friday,1,20260109
2026-01-10,2026,1,January,10,Saturday,1,20260110


## Creating the FactFlight table

### Load the three dimension tables

Before joining, we need the dimension tables loaded as DataFrames so we 
can match Silver's flight rows against them.

In [0]:
dim_airline_df = spark.table("aviation_ws.gold.dim_airline")
dim_airport_df = spark.table("aviation_ws.gold.dim_airport")
dim_date_df = spark.table("aviation_ws.gold.dim_date")

### Join against DimAirline

We match each flight's OP_UNIQUE_CARRIER against DimAirline's CarrierCode 
to bring in AirlineKey.

In [0]:
fact_step1 = df_silver_flight.join(dim_airline_df,df_silver_flight["OP_UNIQUE_CARRIER"]==dim_airline_df["Carrier_Code"],"left")
fact_step1.select("OP_UNIQUE_CARRIER","AirlineKey").limit(10).display()

OP_UNIQUE_CARRIER,AirlineKey
DL,4
DL,4
DL,4
DL,4
DL,4
DL,4
DL,4
DL,4
DL,4
DL,4


### Join against DimAirport for ORIGIN

We need the airport table's columns renamed first — otherwise this join 
and the next one (for DEST) would both produce a column called 
"AirportKey", which Spark can't tell apart.

In [0]:
dim_airport_origin = (dim_airport_df.withColumnRenamed("AirportKey", "OriginAirportKey")
                                    .withColumnRenamed("AirportCode", "OriginAirportCode"))


In [0]:
fact_step2 = fact_step1.join(dim_airport_origin,fact_step1["ORIGIN"] == dim_airport_origin["OriginAirportCode"],"left")
fact_step2.select("ORIGIN", "OriginAirportKey").limit(10).display()

ORIGIN,OriginAirportKey
JAX,168
LAX,181
ATL,18
SEA,292
ANC,15
ATL,18
LAS,179
BZN,55
SEA,292
LAX,181


### Join against DimAirport again, this time for DEST

Same table, same rename technique, but aliased for the destination side 
this time.

In [0]:
dim_airport_dest = (dim_airport_df.withColumnRenamed("AirportKey", "DestAirportKey")
                                    .withColumnRenamed("AirportCode", "DestAirportCode"))

In [0]:
fact_step3 = fact_step2.join(dim_airport_dest,fact_step2["DEST"] == dim_airport_dest["DestAirportCode"],"left")
fact_step3.select("DEST", "DestAirportKey").limit(10).display()

DEST,DestAirportKey
LGA,190
MSP,224
DFW,89
JFK,169
ATL,18
ANC,15
ATL,18
BOS,42
HNL,147
DFW,89


### Join against DimDate

Match each flight's FL_DATE against DimDate's FullDate to bring in 
DateKey.


In [0]:
fact_base = fact_step3.join(dim_date_df,fact_step3["FL_DATE"] == dim_date_df["FullDate"],"left")
fact_base.select("FL_DATE", "DateKey").limit(10).display()

FL_DATE,DateKey
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102
2026-01-02,20260102


In [0]:
fact_base.select("FL_DATE", "DateKey","OP_UNIQUE_CARRIER", "AirlineKey","ORIGIN", "OriginAirportKey","DEST", "DestAirportKey"
).limit(10).display()

FL_DATE,DateKey,OP_UNIQUE_CARRIER,AirlineKey,ORIGIN,OriginAirportKey,DEST,DestAirportKey
2026-01-02,20260102,DL,4,JAX,168,LGA,190
2026-01-02,20260102,DL,4,LAX,181,MSP,224
2026-01-02,20260102,DL,4,ATL,18,DFW,89
2026-01-02,20260102,DL,4,SEA,292,JFK,169
2026-01-02,20260102,DL,4,ANC,15,ATL,18
2026-01-02,20260102,DL,4,ATL,18,ANC,15
2026-01-02,20260102,DL,4,LAS,179,ATL,18
2026-01-02,20260102,DL,4,BZN,55,BOS,42
2026-01-02,20260102,DL,4,SEA,292,HNL,147
2026-01-02,20260102,DL,4,LAX,181,DFW,89


In [0]:
fact_base.filter(col("AirlineKey").isNull()).count()
fact_base.filter(col("OriginAirportKey").isNull()).count()
fact_base.filter(col("DestAirportKey").isNull()).count()
fact_base.filter(col("DateKey").isNull()).count()

0

### Build the final FactFlight column set

Now that fact_base has all four dimension keys correctly joined in, we 
drop the raw text columns we no longer need (carrier code, airport codes, 
raw date) since that information now lives in the dimensions and is 
reachable via the keys. What remains is: the four keys, the flight's 
measures (delays, times, distance), and the two hash columns carried over 
from Silver.

In [0]:
fact_flight_final = fact_base.select(col("DateKey"),
                                    col("AirlineKey"),
                                    col("OriginAirportKey"),
                                    col("DestAirportKey"),
                                    col("OP_CARRIER_FL_NUM").alias("FlightNumber"),
                                    col("CRS_DEP_TIME"),
                                    col("DEP_TIME"),
                                    col("DEP_DELAY"),
                                    col("CRS_ARR_TIME"),
                                    col("ARR_TIME"),
                                    col("ARR_DELAY"),
                                    col("CANCELLED"),
                                    col("DIVERTED"),
                                    col("CANCELLATION_CODE"),
                                    col("CRS_ELAPSED_TIME"),
                                    col("ACTUAL_ELAPSED_TIME"),
                                    col("AIR_TIME"),
                                    col("DISTANCE"),
                                    col("CARRIER_DELAY"),
                                    col("WEATHER_DELAY"),
                                    col("NAS_DELAY"),
                                    col("SECURITY_DELAY"),
                                    col("LATE_AIRCRAFT_DELAY"),
                                    col("FlightKeyHash"),
                                    col("RowChangedHash"))

fact_flight_final.limit(10).display()

DateKey,AirlineKey,OriginAirportKey,DestAirportKey,FlightNumber,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,DIVERTED,CANCELLATION_CODE,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,FlightKeyHash,RowChangedHash
20260102,4,168,190,412,1205,1234,29.00,1418,1439,21.00,false,false,null,133.00,125.00,108.00,834.00,0.00,0.00,0.00,0.00,21.00,00b54af791e849e9590d8a06355b81df0269c4845bee120577179913473910ed,86006f721f0b6f2f89cd4552323e31dd722a0fd5ab09309bf1c5dc85fb81412d
20260102,4,181,224,413,1155,1146,-9.00,1735,1723,-12.00,false,false,null,220.00,217.00,190.00,1535.00,null,null,null,null,null,42a19c0d1195b2a4b316b0fe3f30105ef232039f66ff97ac5a6379fda203b839,7833ce3bf6e83392c28f7589a25c4ff84e93822265d41befef45cddb2f53a3dc
20260102,4,18,89,414,1100,1141,41.00,1231,1301,30.00,false,false,null,151.00,140.00,112.00,731.00,30.00,0.00,0.00,0.00,0.00,1502d6359fb809206080d303f10121dbca5610ac9bb4786fb48427d4d4c40ab5,0c018f94b5fdd5e9dd1f2ddd65ad25b7a683863198303daad1e998cdd4861a4a
20260102,4,292,169,415,1136,1218,42.00,2007,2039,32.00,false,false,null,331.00,321.00,276.00,2422.00,1.00,0.00,0.00,0.00,31.00,9ad727283558ad9f5cf04d2528f8b1c53fdfaa6f0fbc91c7176c35abd85aaac0,d5f13bf6fa0db6328710bbdefe1b05f009cb7be363fcb7f9e0e43bf9a7247e6b
20260102,4,15,18,416,2015,2052,37.00,718,741,23.00,false,false,null,423.00,409.00,365.00,3417.00,23.00,0.00,0.00,0.00,0.00,43d6919f5fd79e2199167fb1dc026b46f5ae331b09e6ee7ceecf35fc05527e28,fc482f122683060804b6ffad775302178267ea8a71f5af993070612d7acefb3a
20260102,4,18,15,416,1515,1513,-2.00,1857,1853,-4.00,false,false,null,462.00,460.00,436.00,3417.00,null,null,null,null,null,e203eadcf91b4d6aa76d1f26639c28c5e5406f5772917de33b70b7feecb75027,b52819ec1a8daf5edbf4fc34c93afb63c73f81f997e82b13966989a15579a3c2
20260102,4,179,18,417,2204,2203,-1.00,500,502,2.00,false,false,null,236.00,239.00,191.00,1747.00,null,null,null,null,null,e1fd6b53e8549a01d4cb0731977263fde1caccb07a86fbe92894a273ca28b7be,ab959352b7c6884e88e12517791d3fd3f25323de597aaf2f3ab07b2925be1cdf
20260102,4,55,42,418,1325,1325,0.00,1955,1926,-29.00,false,false,null,270.00,241.00,214.00,1991.00,null,null,null,null,null,3cd86ef9c70dfc4255ac48e743f0e7ef7dba20f99f1a7f9049f51bfa69f96af9,03ea5307d0a151a29c2af182db0b84934c7f3857f878255222676a7fd95349d6
20260102,4,292,147,419,1730,1727,-3.00,2200,2125,-35.00,false,false,null,390.00,358.00,338.00,2677.00,null,null,null,null,null,b83d726c45722e465e885fb348c21a095687e14ca30b5662850966546e0f794a,3fcafcf7d27b55ea3b6b6fcc3879e9bac0eed2040c11d65149ea13a361091b9b
20260102,4,181,89,420,1830,1848,18.00,2329,2338,9.00,false,false,null,179.00,170.00,143.00,1235.00,null,null,null,null,null,7ecb8571e3c3db11b6a27b1cf636c2cde0400955f2671b5c366c9cc10b6f9dbe,0f21216dc1d4ff47fcf60e0b2336fc281e22c0886e051eeaf92099fd1ec665f0


### Verify row count matches Silver

If Section 1's joins were clean, FactFlight should have exactly the same 
number of rows as Silver — one row per flight, nothing added or dropped.

In [0]:
fact_row_count = fact_flight_final.count()
silver_row_count = df_silver_flight.count()

print(f"FactFlight row count: {fact_row_count}")
print(f"Silver row count: {silver_row_count}")
print(f"Match: {fact_row_count == silver_row_count}")

FactFlight row count: 544003
Silver row count: 544003
Match: True


### Check if FactFlight already exists if not than Create FactFlight else perform Merge operation

If the table doesn't exist, we write it directly. If it does exist, we use 
the same MERGE logic as Silver: match incoming rows to existing rows by 
FlightKeyHash, update only if RowChangedHash differs, insert if there's no 
match at all.

In [0]:
fact_table_exists = spark.catalog.tableExists("aviation_ws.gold.fact_flight")

if not fact_table_exists:
    fact_flight_final.write.format("delta").saveAsTable("aviation_ws.gold.fact_flight")
    print("FactFlight table created")
else:
    fact_table = DeltaTable.forName(spark,"aviation_ws.gold.fact_flight")
    fact_table.alias("target").merge(fact_flight_final.alias("source"),"target.FlightKeyHash = source.FlightKeyHash")\
                            .whenMatchedUpdateAll(condition = "target.RowChangedHash !=source.RowChangedHash")\
                            .whenNotMatchedInsertAll()\
                            .execute()
    print("Merge Complete")

FactFlight table created


In [0]:
spark.sql("SELECT COUNT(*) FROM aviation_ws.gold.fact_flight").show()
spark.table("aviation_ws.gold.fact_flight").limit(10).display()

+--------+
|COUNT(*)|
+--------+
|  544003|
+--------+

+--------+----------+----------------+--------------+------------+------------+--------+---------+------------+--------+---------+---------+--------+-----------------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+--------------------+--------------------+
| DateKey|AirlineKey|OriginAirportKey|DestAirportKey|FlightNumber|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|DIVERTED|CANCELLATION_CODE|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|       FlightKeyHash|      RowChangedHash|
+--------+----------+----------------+--------------+------------+------------+--------+---------+------------+--------+---------+---------+--------+-----------------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------